# Workshop walkthrough — group appeals

**COMPTEXT 2026 · University of Birmingham**

An end-to-end pass through the `groupappeals` pipeline on the workshop sample data. Open the [codebook](../docs/codebook.md) in another tab as you work.

## 1. Inspect the input

In [ ]:
import pandas as pd
df_in = pd.read_csv('../data/sample_manifestos.csv')
print(df_in.shape)
df_in.head()

## 2. Run the full pipeline

In [ ]:
from groupappeals import run_full_pipeline

result = run_full_pipeline(
    input_file='../data/sample_manifestos.csv',
    output_file='../sample_results.csv',
    text_column='text',
    create_composite_id=['party', 'year', 'sentence_id'],
    clean_labels=True,
    split_groups=True,
    batch_size=8,
)
result.head()

## 3. Quick distribution checks

In [ ]:
has_group = result['Exact.Group.Text'].notna()
print('with group:', has_group.sum(), '/', len(result))
result.loc[has_group, 'Stance_Clean'].value_counts()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

stance_by_party = (
    result[has_group]
    .assign(party=lambda d: d['text_id'].str.split('_').str[0])
    .groupby(['party', 'Stance_Clean']).size().unstack(fill_value=0)
)
stance_by_party.plot(kind='bar', stacked=True)
plt.title('Stance distribution by party (sample data)')
plt.tight_layout()
plt.show()

## 4. Spot-check

Inspect a few rows by hand. Where does the model agree with your manual annotations from Exercise A? Where does it diverge?

In [ ]:
cols = ['text', 'Exact.Group.Text', 'Stance_Clean', 'Policy_Clean', 'Group1']
result[has_group][cols].sample(min(10, has_group.sum()), random_state=42)